In [4]:
# %pip install aieng-agents

In [2]:
from aieng.agents.tools import CodeInterpreter
from pathlib import Path

# --- CONFIGURATION ---
DB_PATH = "/data"
AGENT_LLM_NAMES = {
    "worker": "gemini-2.5-flash",
    "manager": "gemini-2.5-pro",
}

# --- SHARED TOOLS ---
local_db = Path(DB_PATH)
shared_code_interpreter = CodeInterpreter(
    template_name="lobsuu8phhb16r4r04yr", 
    # template_name="q1sg157kmhnqbfjth0ue", 
    local_files=[local_db] if local_db.exists() else []
)

In [3]:
from aieng.agents import setup_langfuse_tracer, set_up_logging
from dotenv import load_dotenv

load_dotenv()
set_up_logging()

# Setup tracing
tracer = setup_langfuse_tracer(service_name="my_agent_app")

# Your agent code here - traces will automatically be sent to Langfuse

In [4]:
result = await shared_code_interpreter.run_code(
    code="""
import os
for root, dirs, files in os.walk('/data'):
    print(f"Path: {root}")
    print(f"Files: {files}")
"""
)
print(result)

{"stdout":["Path: /data","Files: ['data.db']"],"stderr":[],"results":null,"error":null}


### Agent for SQLite db file

In [5]:
import json
from typing import Any

import sqlglot
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types


E2B_SQLITE_PATH = "/data/data.db"
MAX_SQL_ROWS = 50
_ALLOWED_ROOTS = {"select", "union", "paren"}
_FORBIDDEN_NODES = {
    "create",
    "insert",
    "update",
    "delete",
    "drop",
    "alter",
    "truncate_table",
    "merge",
    "command",
    "pragma",
    "attach",
    "detach",
    "set",
}


def _normalize_code_interpreter_result(raw_result: Any) -> dict[str, Any]:
    if isinstance(raw_result, str):
        return json.loads(raw_result)
    if isinstance(raw_result, dict):
        return raw_result
    if hasattr(raw_result, "model_dump"):
        return raw_result.model_dump()
    raise TypeError(f"Unsupported E2B result type: {type(raw_result)!r}")


def _extract_stdout_text(raw_result: Any) -> str:
    payload = _normalize_code_interpreter_result(raw_result)
    error = payload.get("error")
    if error:
        raise RuntimeError(f"E2B execution failed: {error}")

    stdout = payload.get("stdout", [])
    if isinstance(stdout, list):
        return "\n".join(str(line) for line in stdout)
    return str(stdout)


def _validate_read_only_sql(query: str) -> str:
    cleaned_query = query.strip().rstrip(";")
    if not cleaned_query:
        raise ValueError("SQL query must not be empty.")

    expressions = sqlglot.parse(cleaned_query, read="sqlite")
    if len(expressions) != 1:
        raise ValueError("Only a single SQL statement is allowed.")

    expression = expressions[0]
    if expression.key.lower() not in _ALLOWED_ROOTS:
        raise ValueError("Only read-only SELECT queries are allowed.")

    for node in expression.walk():
        if node.key.lower() in _FORBIDDEN_NODES:
            raise ValueError(f"Forbidden SQL operation detected: {node.key}")

    return cleaned_query


async def get_database_schema() -> str:
    """Return tables and columns available in the E2B SQLite database."""
    code = f"""
import sqlite3

conn = sqlite3.connect({E2B_SQLITE_PATH!r})
rows = conn.execute(
    \"\"\"
    SELECT name, type
    FROM sqlite_master
    WHERE type IN ('table', 'view')
      AND name NOT LIKE 'sqlite_%'
    ORDER BY type, name
    \"\"\"
).fetchall()

sections = []
for name, relation_type in rows:
    table_info = conn.execute(f'PRAGMA table_info("{{name}}")').fetchall()
    columns = ', '.join(f"{{column[1]}} {{column[2] or 'TEXT'}}" for column in table_info)
    sections.append(f"{{relation_type}}: {{name}}\\n  columns: {{columns}}")

print('\\n\\n'.join(sections) or 'No tables found.')
conn.close()
"""
    raw_result = await shared_code_interpreter.run_code(code=code)
    return _extract_stdout_text(raw_result)


async def run_sql_query(query: str) -> str:
    """Execute a read-only SQL query against the E2B SQLite database."""
    safe_query = _validate_read_only_sql(query)
    code = f"""
import json
import sqlite3

query = {safe_query!r}
conn = sqlite3.connect({E2B_SQLITE_PATH!r})
conn.row_factory = sqlite3.Row
cursor = conn.execute(query)
columns = [column[0] for column in (cursor.description or [])]
rows = [dict(row) for row in cursor.fetchmany({MAX_SQL_ROWS})]
payload = {{
    'columns': columns,
    'rows': rows,
    'row_limit': {MAX_SQL_ROWS},
    'query': query,
}}
print(json.dumps(payload, default=str))
conn.close()
"""
    raw_result = await shared_code_interpreter.run_code(code=code)
    return _extract_stdout_text(raw_result)


SQL_AGENT_INSTRUCTIONS = f"""
You are a lightweight SQL analyst for a SQLite database stored in an E2B sandbox at {E2B_SQLITE_PATH}.

Core behavior:
- When the user asks a data question in natural language, automatically inspect schema if needed, write SQL, execute it with run_sql_query, and return the answer.
- Only skip execution if the user explicitly asks for SQL only, query only, or no execution.
- Never invent table or column names.
- Only use read-only SQL.
- Keep answers concise and grounded in tool output.

Response format for normal data questions:
ANSWER:
- Give the direct answer first in plain English.

SQL:
```sql
<the exact SQL you executed>
```

RESULT:
- Summarize the returned rows clearly.
- Mention when results are capped at {MAX_SQL_ROWS} rows.

If execution fails:
- Explain the failure briefly.
- Fix the SQL and try again.
""".strip()

sql_agent = Agent(
    name="e2b_sql_agent",
    model=AGENT_LLM_NAMES["worker"],
    instruction=SQL_AGENT_INSTRUCTIONS,
    tools=[get_database_schema, run_sql_query],
)

sql_session_service = InMemorySessionService()
sql_runner = Runner(
    app_name=sql_agent.name,
    agent=sql_agent,
    session_service=sql_session_service,
)
sql_session = await sql_session_service.create_session(
    app_name=sql_agent.name,
    user_id="notebook-user",
    state={},
)


async def reset_sql_agent_session() -> str:
    """Start a fresh conversation with the SQL agent."""
    global sql_session
    sql_session = await sql_session_service.create_session(
        app_name=sql_agent.name,
        user_id="notebook-user",
        state={},
    )
    return sql_session.id


async def ask_sql_agent(question: str) -> str:
    """Ask the E2B-backed SQL agent a natural-language question and get the executed answer."""
    content = types.Content(role="user", parts=[types.Part(text=question)])
    final_chunks: list[str] = []

    async for event in sql_runner.run_async(
        user_id="notebook-user",
        session_id=sql_session.id,
        new_message=content,
    ):
        if not event.is_final_response() or not event.content:
            continue
        for part in event.content.parts:
            if part.text:
                final_chunks.append(part.text)

    return "\n".join(chunk for chunk in final_chunks if chunk).strip()


async def answer_query(question: str) -> str:
    """Convenience wrapper for natural-language database questions."""
    return await ask_sql_agent(question)


print(f"SQL agent ready for {E2B_SQLITE_PATH}. Session: {sql_session.id}")

SQL agent ready for /data/data.db. Session: 7f216ab0-4e11-4172-a212-5f2e15a9acd3


In [6]:
schema_preview = await get_database_schema()
print(schema_preview[:4000])

table: cards
  columns: card_id INTEGER, client_id INTEGER, card_brand TEXT, card_type TEXT, card_number INTEGER, expires TEXT, cvv INTEGER, has_chip TEXT, num_cards_issued INTEGER, credit_limit TEXT, acct_open_date TEXT, year_pin_last_changed INTEGER, card_on_dark_web TEXT

table: fraud_labels
  columns: transaction_id INTEGER, fraud TEXT

table: mcc_codes
  columns: mcc INTEGER, mcc_description TEXT

table: transactions
  columns: transaction_id INTEGER, date TEXT, client_id INTEGER, card_id INTEGER, amount TEXT, use_chip TEXT, merchant_id INTEGER, merchant_city TEXT, merchant_state TEXT, zip REAL, mcc INTEGER, errors TEXT

table: users
  columns: client_id INTEGER, current_age INTEGER, retirement_age INTEGER, birth_year INTEGER, birth_month INTEGER, gender TEXT, address TEXT, latitude REAL, longitude REAL, per_capita_income TEXT, yearly_income TEXT, total_debt TEXT, credit_score INTEGER, num_credit_cards INTEGER


In [11]:
response = await answer_query(
    # "What are the top 5 merchant states by transaction count?"
    "Scan the database for high-frequency transaction fraud. return the client id and the transaction id."
)
print(response)

16:36:24.108 invocation
16:36:24.110   invoke_agent e2b_sql_agent
16:36:24.111     call_llm
16:36:25.462       execute_tool run_sql_query


16:36:29.494     call_llm
ANSWER:
The following are client IDs and their associated fraudulent transaction IDs (capped at 50 rows):
- client_id 3, transaction_id 13458759
- client_id 3, transaction_id 13458910
- client_id 3, transaction_id 13463010
- client_id 3, transaction_id 13463066
- client_id 3, transaction_id 13463759
- client_id 3, transaction_id 13467717
- client_id 3, transaction_id 13468320
- client_id 3, transaction_id 13470400
- client_id 3, transaction_id 13474999
- client_id 3, transaction_id 16019603
- client_id 3, transaction_id 16020183
- client_id 3, transaction_id 16021384
- client_id 3, transaction_id 16267414
- client_id 3, transaction_id 16267451
- client_id 3, transaction_id 16270580
- client_id 3, transaction_id 16270866
- client_id 3, transaction_id 16270872
- client_id 13, transaction_id 18336476
- client_id 13, transaction_id 21497910
- client_id 13, transaction_id 21507538
- client_id 13, transaction_id 21508727
- client_id 17, transaction_id 17767008
- cli

In [14]:
import csv
import json
from pathlib import Path


# Analyze fraudulent transactions for IDs loaded from CSV only.
CSV_CANDIDATES = [
    Path("dataset_input_ids.csv"),
    Path("implementations/cibc-three/dataset_input_ids.csv"),
]


def _find_csv_path() -> Path:
    for candidate in CSV_CANDIDATES:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find CSV file. Tried: {[str(p) for p in CSV_CANDIDATES]}"
    )


def _as_int_list(values):
    out = []
    for value in values:
        text = str(value).strip()
        if not text:
            continue
        try:
            out.append(int(text))
        except ValueError:
            continue
    return out


def _chunks(values, size):
    for i in range(0, len(values), size):
        yield values[i : i + size]


async def _run_sql_json(query: str) -> dict:
    return json.loads(await run_sql_query(query))


async def _pick_existing_column(table: str, candidates: list[str]) -> str:
    for col in candidates:
        try:
            await run_sql_query(f"SELECT {col} FROM {table} LIMIT 1")
            return col
        except Exception:
            continue
    raise RuntimeError(f"None of these columns exist in {table}: {candidates}")


csv_path = _find_csv_path()
with csv_path.open("r", newline="") as f:
    raw_lines = [line.strip() for line in f if line.strip()]

if not raw_lines:
    raise ValueError(f"CSV file is empty: {csv_path}")

first_token = raw_lines[0].split(",")[0].strip().lower()
known_headers = {"client_id", "transaction_id", "id"}

# Support both formats:
# 1) header + values (client_id/transaction_id/id)
# 2) values only (one ID per row, no header) -> treated as client IDs
if first_token in known_headers:
    with csv_path.open("r", newline="") as f:
        reader = csv.DictReader(f)
        fieldnames = [name.strip() for name in (reader.fieldnames or [])]
        id_column = "client_id" if "client_id" in fieldnames else fieldnames[0]
        source_ids = [row.get(id_column, "") for row in reader]
else:
    id_column = "client_id"
    source_ids = [line.split(",")[0].strip() for line in raw_lines]

source_ids = _as_int_list(source_ids)
if not source_ids:
    raise ValueError(f"No numeric IDs found in {csv_path}.")

transactions_client_col = await _pick_existing_column(
    "transactions", ["client_id", "clientid", "user_id", "customer_id"]
)
transactions_tx_col = await _pick_existing_column(
    "transactions", ["transaction_id", "tx_id", "id", "transactionid"]
)

# If CSV has transaction IDs, map them to client IDs in the DB.
if id_column == "client_id":
    client_ids = sorted(set(source_ids))
else:
    mapped_client_ids = set()
    for batch in _chunks(source_ids, 300):
        in_clause = ",".join(str(v) for v in batch)
        map_query = (
            f"SELECT DISTINCT {transactions_client_col} AS client_id "
            f"FROM transactions WHERE {transactions_tx_col} IN ({in_clause})"
        )
        payload = await _run_sql_json(map_query)
        for row in payload.get("rows", []):
            if row.get("client_id") is not None:
                mapped_client_ids.add(int(row["client_id"]))
    client_ids = sorted(mapped_client_ids)

if not client_ids:
    raise ValueError("No client IDs available after CSV parsing/mapping.")

# Find compatible fraud_labels join + label columns.
label_candidates = ["is_fraud", "fraud", "label", "is_laundering"]
fraud_tx_candidates = ["transaction_id", "tx_id", "id"]

fraud_true_expr_template = (
    "CASE WHEN lower(CAST(f.{label_col} AS TEXT)) "
    "IN ('1','true','yes','y','fraud','fraudulent') THEN 1 ELSE 0 END"
)

chosen = None
for label_col in label_candidates:
    for fraud_tx_col in fraud_tx_candidates:
        try:
            probe_ids = ",".join(str(v) for v in client_ids[:20])
            probe_query = (
                f"SELECT COUNT(*) AS c FROM transactions t "
                f"JOIN fraud_labels f ON f.{fraud_tx_col} = t.{transactions_tx_col} "
                f"WHERE t.{transactions_client_col} IN ({probe_ids})"
            )
            await run_sql_query(probe_query)
            chosen = (label_col, fraud_tx_col)
            break
        except Exception:
            continue
    if chosen is not None:
        break

if chosen is None:
    raise RuntimeError(
        "Could not find compatible join/label columns between transactions and fraud_labels."
    )

label_col, fraud_tx_col = chosen
fraud_true_expr = fraud_true_expr_template.format(label_col=label_col)

total_transactions = 0
total_fraudulent_transactions = 0

for batch in _chunks(client_ids, 300):
    in_clause = ",".join(str(v) for v in batch)
    summary_query = (
        f"SELECT "
        f"COUNT(*) AS total_transactions, "
        f"COALESCE(SUM({fraud_true_expr}), 0) AS fraudulent_transactions "
        f"FROM transactions t "
        f"JOIN fraud_labels f ON f.{fraud_tx_col} = t.{transactions_tx_col} "
        f"WHERE t.{transactions_client_col} IN ({in_clause})"
    )
    payload = await _run_sql_json(summary_query)
    row = (payload.get("rows") or [{}])[0]
    total_transactions += int(row.get("total_transactions") or 0)
    total_fraudulent_transactions += int(row.get("fraudulent_transactions") or 0)

fraud_rate = (
    (total_fraudulent_transactions / total_transactions) if total_transactions else 0.0
)

sample_ids_clause = ",".join(str(v) for v in client_ids[:300])
sample_query = (
    f"SELECT t.{transactions_client_col} AS client_id, t.{transactions_tx_col} AS transaction_id "
    f"FROM transactions t "
    f"JOIN fraud_labels f ON f.{fraud_tx_col} = t.{transactions_tx_col} "
    f"WHERE t.{transactions_client_col} IN ({sample_ids_clause}) "
    f"AND ({fraud_true_expr}) = 1 "
    f"ORDER BY t.{transactions_tx_col} DESC LIMIT 50"
)
sample_payload = await _run_sql_json(sample_query)

print(f"CSV used: {csv_path}")
print(f"ID interpretation used: {id_column}")
print(f"Unique client IDs analyzed: {len(client_ids)}")
print(f"transactions client column: {transactions_client_col}")
print(f"transactions transaction column: {transactions_tx_col}")
print(f"fraud_labels transaction column: {fraud_tx_col}")
print(f"fraud_labels label column: {label_col}")
print(f"Total transactions (for those clients): {total_transactions}")
print(f"Fraudulent transactions: {total_fraudulent_transactions}")
print(f"Fraud rate: {fraud_rate:.2%}")

fraud_sample_rows = sample_payload.get("rows", [])
print(f"Sample fraudulent rows returned: {len(fraud_sample_rows)}")
fraud_sample_rows[:10]

CancelledError: 

In [20]:
query_preview = await run_sql_query(
    "SELECT name, type FROM sqlite_master WHERE type IN ('table', 'view') AND name NOT LIKE 'sqlite_%' ORDER BY type, name"
)
print(query_preview)

{"columns": ["name", "type"], "rows": [], "row_limit": 50, "query": "SELECT name, type FROM sqlite_master WHERE type IN ('table', 'view') AND name NOT LIKE 'sqlite_%' ORDER BY type, name"}


### Forcing the agent to pick from the ID's in the CSV

<!-- table: cards
  columns: card_id INTEGER, client_id INTEGER, card_brand TEXT, card_type TEXT, card_number INTEGER, expires TEXT, cvv INTEGER, has_chip TEXT, num_cards_issued INTEGER, credit_limit TEXT, acct_open_date TEXT, year_pin_last_changed INTEGER, card_on_dark_web TEXT

table: fraud_labels
  columns: transaction_id INTEGER, fraud TEXT

table: mcc_codes
  columns: mcc INTEGER, mcc_description TEXT

table: transactions
  columns: transaction_id INTEGER, date TEXT, client_id INTEGER, card_id INTEGER, amount TEXT, use_chip TEXT, merchant_id INTEGER, merchant_city TEXT, merchant_state TEXT, zip REAL, mcc INTEGER, errors TEXT

table: users -->

In [18]:

import csv
from pathlib import Path

import sqlglot
from sqlglot import expressions as exp


CSV_CANDIDATES = [
    Path("dataset_input_ids.csv"),
    Path("implementations/cibc-three/dataset_input_ids.csv"),
]

# Define exactly which SQLite tables the agent is allowed to use.
ALLOWED_SQL_TABLES = {
    "transactions",
    "cards",
    "mcc_codes",
    "users"
}


def _find_scope_csv_path() -> Path:
    for candidate in CSV_CANDIDATES:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find CSV file. Tried: {[str(p) for p in CSV_CANDIDATES]}"
    )


def _as_int_list(values):
    out = []
    for value in values:
        text = str(value).strip()
        if not text:
            continue
        try:
            out.append(int(text))
        except ValueError:
            continue
    return out


def _load_scoped_transaction_ids(csv_path: Path) -> list[int]:
    with csv_path.open("r", newline="") as f:
        raw_lines = [line.strip() for line in f if line.strip()]

    if not raw_lines:
        raise ValueError(f"CSV file is empty: {csv_path}")

    first_token = raw_lines[0].split(",")[0].strip().lower()
    known_headers = {"transaction_id", "tx_id", "id"}

    if first_token in known_headers:
        with csv_path.open("r", newline="") as f:
            reader = csv.DictReader(f)
            fieldnames = [name.strip() for name in (reader.fieldnames or [])]
            id_column = "transaction_id" if "transaction_id" in fieldnames else fieldnames[0]
            source_ids = [row.get(id_column, "") for row in reader]
    else:
        source_ids = [line.split(",")[0].strip() for line in raw_lines]

    transaction_ids = sorted(set(_as_int_list(source_ids)))
    if not transaction_ids:
        raise ValueError(f"No numeric transaction IDs found in {csv_path}")
    return transaction_ids


async def _pick_transactions_key_column() -> str:
    candidates = ["transaction_id", "tx_id", "id", "transactionid"]
    for col in candidates:
        try:
            await _original_run_sql_query(f"SELECT {col} FROM transactions LIMIT 1")
            return col
        except Exception:
            continue
    raise RuntimeError(f"None of these columns exist in transactions: {candidates}")


def _validate_allowed_tables(query: str, allowed_tables: set[str]) -> str:
    parsed = sqlglot.parse_one(query, read="sqlite")

    # Collect CTE names so they are not mistaken for real tables
    cte_names = {
        str(cte.alias).lower()
        for cte in parsed.find_all(exp.CTE)
        if cte.alias
    }

    referenced_tables = {
        str(table.name).lower()
        for table in parsed.find_all(exp.Table)
        if table.name and str(table.name).lower() not in cte_names
    }

    disallowed = sorted(table for table in referenced_tables if table not in allowed_tables)
    if disallowed:
        raise ValueError(
            "Query references disallowed tables: "
            f"{disallowed}. Allowed tables are: {sorted(allowed_tables)}"
        )
    return query


def _scope_sql_to_transaction_ids(
    query: str,
    transaction_ids: list[int],
    transactions_tx_col: str,
) -> str:
    if not transaction_ids:
        raise ValueError("Transaction ID scope is empty.")

    parsed = sqlglot.parse_one(query, read="sqlite")
    aliases = []
    for table in parsed.find_all(exp.Table):
        if str(table.name).lower() == "transactions":
            aliases.append(table.alias_or_name or str(table.name))

    if not aliases:
        return query

    in_values = ",".join(str(v) for v in transaction_ids)
    alias_conditions = [
        sqlglot.parse_one(f"{alias}.{transactions_tx_col} IN ({in_values})", read="sqlite")
        for alias in aliases
    ]

    scope_condition = alias_conditions[0]
    for extra_condition in alias_conditions[1:]:
        scope_condition = exp.or_(scope_condition, extra_condition)

    if isinstance(parsed, exp.Select):
        where_clause = parsed.args.get("where")
        if where_clause is None:
            parsed.set("where", exp.Where(this=scope_condition))
        else:
            parsed.set("where", exp.Where(this=exp.and_(where_clause.this, scope_condition)))
        return parsed.sql(dialect="sqlite")

    return parsed.sql(dialect="sqlite")


scope_csv_path = _find_scope_csv_path()
SCOPED_TRANSACTION_IDS = _load_scoped_transaction_ids(scope_csv_path)

if "_original_run_sql_query" not in globals():
    _original_run_sql_query = run_sql_query

SCOPED_TRANSACTIONS_TX_COL = await _pick_transactions_key_column()


async def run_sql_query(query: str) -> str:
    safe_query = _validate_read_only_sql(query)
    _validate_allowed_tables(safe_query, ALLOWED_SQL_TABLES)
    scoped_query = _scope_sql_to_transaction_ids(
        safe_query,
        SCOPED_TRANSACTION_IDS,
        SCOPED_TRANSACTIONS_TX_COL,
    )
    return await _original_run_sql_query(scoped_query)


async def get_database_schema() -> str:
    """Return only the allowed tables and their columns to the agent."""
    code = f"""
import sqlite3

conn = sqlite3.connect({E2B_SQLITE_PATH!r})
allowed = {sorted(ALLOWED_SQL_TABLES)!r}
rows = conn.execute(
    \"\"\"
    SELECT name FROM sqlite_master
    WHERE type = 'table' AND name NOT LIKE 'sqlite_%'
    ORDER BY name
    \"\"\"
).fetchall()

sections = []
for (name,) in rows:
    if name not in allowed:
        continue
    table_info = conn.execute(f'PRAGMA table_info("{{name}}")').fetchall()
    columns = ', '.join(f"{{col[1]}} {{col[2] or 'TEXT'}}" for col in table_info)
    sections.append(f"table: {{name}}\\n  columns: {{columns}}")

print('\\n\\n'.join(sections) or 'No tables found.')
conn.close()
"""
    raw_result = await shared_code_interpreter.run_code(code=code)
    return _extract_stdout_text(raw_result)


# SQL_AGENT_INSTRUCTIONS = f"""
# You are a lightweight SQL analyst for a SQLite database stored in an E2B sandbox at {E2B_SQLITE_PATH}.

# Core behavior:
# - When the user asks a data question in natural language, automatically inspect schema if needed, write SQL, execute it with run_sql_query, and return the answer.
# - Only skip execution if the user explicitly asks for SQL only, query only, or no execution.
# - Never invent table or column names. Only use tables returned by get_database_schema.
# - Only use read-only SQL.
# - Keep answers concise and grounded in tool output.
# - CRITICAL: Only analyze rows belonging to the preloaded scoped transaction IDs. Do not reason outside that scope.
# - CRITICAL: Use only these tables: {sorted(ALLOWED_SQL_TABLES)}. CTEs are allowed but must only reference these base tables.

# Response format for normal data questions:
# ANSWER:
# - Give the direct answer first in plain English.

# SQL:
# ```sql
# <the exact SQL you executed>
# ```

# RESULT:
# - Summarize the returned rows clearly.
# - Mention when results are capped at {MAX_SQL_ROWS} rows.

# If execution fails:
# - Explain the failure briefly.
# - Fix the SQL and try again.
# """.strip()

# sql_agent = Agent(
#     name="e2b_sql_agent_scoped_tx",
#     model=AGENT_LLM_NAMES["worker"],
#     instruction=SQL_AGENT_INSTRUCTIONS,
#     tools=[get_database_schema, run_sql_query],
# )

# sql_runner = Runner(
#     app_name=sql_agent.name,
#     agent=sql_agent,
#     session_service=sql_session_service,
# )
# sql_session = await sql_session_service.create_session(
#     app_name=sql_agent.name,
#     user_id="notebook-user",
#     state={},
# )

# print(f"Scoped SQL agent ready. Session: {sql_session.id}")
# print(f"CSV used for scope: {scope_csv_path}")
# print(f"Scoped transaction IDs loaded: {len(SCOPED_TRANSACTION_IDS)}")
# print(f"transactions key column used: {SCOPED_TRANSACTIONS_TX_COL}")
# print(f"Allowed SQL tables: {sorted(ALLOWED_SQL_TABLES)}")


In [19]:
# Reconfigure scoped SQL agent to always call get_database_schema first
# and harden scoped query column detection for transactions key
import json


async def _detect_transactions_key_column_from_schema() -> str:
    """Read transactions columns directly from SQLite and pick the best key column."""
    code = f"""
import json
import sqlite3

conn = sqlite3.connect({E2B_SQLITE_PATH!r})
cols = [row[1] for row in conn.execute("PRAGMA table_info(transactions)").fetchall()]
conn.close()
print(json.dumps(cols))
"""
    raw_result = await shared_code_interpreter.run_code(code=code)
    cols_text = _extract_stdout_text(raw_result).strip()
    cols = json.loads(cols_text)

    candidates = ["transaction_id", "tx_id", "id", "transactionid"]
    for col in candidates:
        if col in cols:
            return col

    raise RuntimeError(
        f"Could not determine transactions key column. Found columns: {cols}"
    )


SCOPED_TRANSACTIONS_TX_COL = await _detect_transactions_key_column_from_schema()
print(f"Scoped transactions key column refreshed: {SCOPED_TRANSACTIONS_TX_COL}")


async def run_sql_query(query: str) -> str:
    """Scoped SQL execution with one-time key-column recovery on schema mismatch."""
    global SCOPED_TRANSACTIONS_TX_COL

    safe_query = _validate_read_only_sql(query)
    _validate_allowed_tables(safe_query, ALLOWED_SQL_TABLES)

    scoped_query = _scope_sql_to_transaction_ids(
        safe_query,
        SCOPED_TRANSACTION_IDS,
        SCOPED_TRANSACTIONS_TX_COL,
    )

    try:
        return await _original_run_sql_query(scoped_query)
    except Exception as exc:
        message = str(exc)
        if "no such column" in message and "transactions." in message:
            # Recover from stale key-column inference and retry once.
            SCOPED_TRANSACTIONS_TX_COL = await _detect_transactions_key_column_from_schema()
            scoped_query = _scope_sql_to_transaction_ids(
                safe_query,
                SCOPED_TRANSACTION_IDS,
                SCOPED_TRANSACTIONS_TX_COL,
            )
            return await _original_run_sql_query(scoped_query)
        raise


SQL_AGENT_INSTRUCTIONS = f"""
You are a lightweight SQL analyst for a SQLite database stored in an E2B sandbox at {E2B_SQLITE_PATH}.

Core behavior:
- For every new user question, ALWAYS call get_database_schema first before writing or executing SQL.
- After reading schema, write SQL, execute it with run_sql_query, and return the answer.
- Only skip execution if the user explicitly asks for SQL only, query only, or no execution.
- Never invent table or column names. Only use tables returned by get_database_schema.
- Only use read-only SQL.
- Keep answers concise and grounded in tool output.
- CRITICAL: Only analyze rows belonging to the preloaded scoped transaction IDs. Do not reason outside that scope.
- CRITICAL: Use only these tables: {sorted(ALLOWED_SQL_TABLES)}. CTEs are allowed but must only reference these base tables.

Response format for normal data questions:
ANSWER:
- Give the direct answer first in plain English.

SQL:
```sql
<the exact SQL you executed>
```

RESULT:
- Summarize the returned rows clearly.
- Mention when results are capped at {MAX_SQL_ROWS} rows.

If execution fails:
- Explain the failure briefly.
- Fix the SQL and try again.
""".strip()

sql_agent = Agent(
    name="e2b_sql_agent_scoped_tx",
    model=AGENT_LLM_NAMES["worker"],
    instruction=SQL_AGENT_INSTRUCTIONS,
    tools=[get_database_schema, run_sql_query],
)

sql_runner = Runner(
    app_name=sql_agent.name,
    agent=sql_agent,
    session_service=sql_session_service,
)
sql_session = await sql_session_service.create_session(
    app_name=sql_agent.name,
    user_id="notebook-user",
    state={},
)

print(f"Scoped SQL agent reconfigured. Session: {sql_session.id}")

Scoped transactions key column refreshed: transaction_id
Scoped SQL agent reconfigured. Session: 01d24cb9-b96f-4124-9f5e-46cc2ee7a8a3


In [23]:
# CSV-scoped SQL helper for E2B: only rows whose transaction IDs are in the CSV are returned
import csv
import json
from pathlib import Path

import sqlglot
from sqlglot import expressions as exp


CSV_SCOPE_CANDIDATES = [
    Path("dataset_input_ids.csv"),
    Path("implementations/cibc-three/dataset_input_ids.csv"),
]


def _find_tx_csv_path() -> Path:
    for candidate in CSV_SCOPE_CANDIDATES:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find CSV file. Tried: {[str(p) for p in CSV_SCOPE_CANDIDATES]}"
    )


def _load_transaction_ids_from_csv(csv_path: Path) -> list[int]:
    with csv_path.open("r", newline="") as f:
        raw_lines = [line.strip() for line in f if line.strip()]

    if not raw_lines:
        raise ValueError(f"CSV file is empty: {csv_path}")

    first_token = raw_lines[0].split(",")[0].strip().lower()
    known_headers = {"transaction_id", "tx_id", "id"}

    if first_token in known_headers:
        with csv_path.open("r", newline="") as f:
            reader = csv.DictReader(f)
            fieldnames = [name.strip() for name in (reader.fieldnames or [])]
            id_column = "transaction_id" if "transaction_id" in fieldnames else fieldnames[0]
            source_ids = [row.get(id_column, "") for row in reader]
    else:
        source_ids = [line.split(",")[0].strip() for line in raw_lines]

    out = []
    for value in source_ids:
        text = str(value).strip()
        if not text:
            continue
        try:
            out.append(int(text))
        except ValueError:
            continue

    tx_ids = sorted(set(out))
    if not tx_ids:
        raise ValueError(f"No numeric transaction IDs found in {csv_path}")
    return tx_ids


async def _detect_transactions_key_column() -> str:
    code = f"""
import json
import sqlite3

conn = sqlite3.connect({E2B_SQLITE_PATH!r})
cols = [row[1] for row in conn.execute('PRAGMA table_info(transactions)').fetchall()]
conn.close()
print(json.dumps(cols))
"""
    raw_result = await shared_code_interpreter.run_code(code=code)
    cols = json.loads(_extract_stdout_text(raw_result).strip())

    for candidate in ["transaction_id", "tx_id", "id", "transactionid"]:
        if candidate in cols:
            return candidate
    raise RuntimeError(f"Could not infer transactions key column. Found: {cols}")


def _scope_query_to_tx_ids(query: str, tx_ids: list[int], tx_col: str) -> str:
    parsed = sqlglot.parse_one(query, read="sqlite")

    aliases = []
    for table in parsed.find_all(exp.Table):
        if str(table.name).lower() == "transactions":
            aliases.append(table.alias_or_name or str(table.name))

    if not aliases:
        raise ValueError("Query must reference the transactions table so ID scoping can be applied.")

    in_values = ",".join(str(v) for v in tx_ids)
    alias_conditions = [
        sqlglot.parse_one(f"{alias}.{tx_col} IN ({in_values})", read="sqlite")
        for alias in aliases
    ]

    scope_condition = alias_conditions[0]
    for extra in alias_conditions[1:]:
        scope_condition = exp.or_(scope_condition, extra)

    if isinstance(parsed, exp.Select):
        where_clause = parsed.args.get("where")
        if where_clause is None:
            parsed.set("where", exp.Where(this=scope_condition))
        else:
            parsed.set("where", exp.Where(this=exp.and_(where_clause.this, scope_condition)))
        return parsed.sql(dialect="sqlite")

    return parsed.sql(dialect="sqlite")


async def run_csv_scoped_query(query: str, row_limit: int = 200) -> dict:
    """Run read-only SQL in E2B while forcing transaction_id scope to IDs from CSV."""
    safe_query = _validate_read_only_sql(query)

    csv_path = _find_tx_csv_path()
    tx_ids = _load_transaction_ids_from_csv(csv_path)
    tx_col = await _detect_transactions_key_column()
    scoped_query = _scope_query_to_tx_ids(safe_query, tx_ids, tx_col)

    code = f"""
import json
import sqlite3

query = {scoped_query!r}
conn = sqlite3.connect({E2B_SQLITE_PATH!r})
conn.row_factory = sqlite3.Row
cursor = conn.execute(query)
columns = [c[0] for c in (cursor.description or [])]
rows = [dict(r) for r in cursor.fetchmany({int(row_limit)})]
payload = {{
    'query': query,
    'columns': columns,
    'rows': rows,
    'row_limit': {int(row_limit)},
}}
print(json.dumps(payload, default=str))
conn.close()
"""

    raw_result = await shared_code_interpreter.run_code(code=code)
    return {
        "csv_path": str(csv_path),
        "scoped_transaction_ids": len(tx_ids),
        "transactions_key_column": tx_col,
        **json.loads(_extract_stdout_text(raw_result).strip()),
    }


# Example usage
scoped_payload = await run_csv_scoped_query(
    """
    SELECT t.transaction_id, t.client_id, t.amount, t.merchant_state, t.mcc
    FROM transactions t
    ORDER BY t.date DESC
    """,
    row_limit=20,
)

print(f"CSV used: {scoped_payload['csv_path']}")
print(f"Scoped IDs loaded: {scoped_payload['scoped_transaction_ids']}")
print(f"transactions key column: {scoped_payload['transactions_key_column']}")
print(f"Rows returned: {len(scoped_payload['rows'])}")
scoped_payload['rows'][:5]

CSV used: dataset_input_ids.csv
Scoped IDs loaded: 15903
transactions key column: transaction_id
Rows returned: 20


[{'transaction_id': 23761868,
  'client_id': 1718,
  'amount': '$1.11',
  'merchant_state': 'CA',
  'mcc': 5499},
 {'transaction_id': 23757877,
  'client_id': 1591,
  'amount': '$-60.00',
  'merchant_state': 'WA',
  'mcc': 5499},
 {'transaction_id': 23756275,
  'client_id': 156,
  'amount': '$-87.00',
  'merchant_state': 'CA',
  'mcc': 5541},
 {'transaction_id': 23755655,
  'client_id': 1452,
  'amount': '$12.82',
  'merchant_state': 'TX',
  'mcc': 5541},
 {'transaction_id': 23755416,
  'client_id': 1846,
  'amount': '$167.64',
  'merchant_state': 'KS',
  'mcc': 3066}]

### Agent Dealing with RAW Files

In [ ]:
from aieng.agents.tools import CodeInterpreter
from pathlib import Path

# --- CONFIGURATION ---
DB_PATH = "/data"
AGENT_LLM_NAMES = {
    "worker": "gemini-2.5-flash",
    "manager": "gemini-2.5-pro",
}

# --- SHARED TOOLS ---
local_db = Path(DB_PATH)
shared_code_interpreter = CodeInterpreter(
    # template_name="lobsuu8phhb16r4r04yr", 
    template_name="q1sg157kmhnqbfjth0ue", 
    local_files=[local_db] if local_db.exists() else []
)

In [21]:
# Template-aware agent for the new E2B file bundle.
# This cell does not modify the existing SQL agent; it adds a new one that can
# inspect all files currently available in the template and run a fraud-focused
# analysis over the raw files.

import json
from pathlib import PurePosixPath
from typing import Any

from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types


_TEMPLATE_ROOT = "/data"
_SUPPORTED_PREVIEW_SUFFIXES = {
    ".csv",
    ".json",
    ".jsonl",
    ".parquet",
    ".txt",
    ".md",
    ".sql",
    ".db",
    ".sqlite",
    ".sqlite3",
}


def _stdout_from_ci_result(raw_result: Any) -> str:
    if isinstance(raw_result, str):
        payload = json.loads(raw_result)
    elif isinstance(raw_result, dict):
        payload = raw_result
    elif hasattr(raw_result, "model_dump"):
        payload = raw_result.model_dump()
    else:
        raise TypeError(f"Unsupported CodeInterpreter result type: {type(raw_result)!r}")

    if payload.get("error"):
        raise RuntimeError(f"E2B execution failed: {payload['error']}")

    stdout = payload.get("stdout", [])
    if isinstance(stdout, list):
        return "\n".join(str(line) for line in stdout)
    return str(stdout)


async def list_template_files() -> str:
    """List files available in the current E2B template under /data."""
    code = f"""
import json
from pathlib import Path

root = Path({_TEMPLATE_ROOT!r})
if not root.exists():
    print(json.dumps({{"root": str(root), "files": [], "note": "root not found"}}))
else:
    files = []
    for path in sorted(root.rglob('*')):
        if path.is_file():
            files.append({{
                "path": str(path),
                "name": path.name,
                "suffix": path.suffix.lower(),
                "size_bytes": path.stat().st_size,
            }})
    print(json.dumps({{"root": str(root), "files": files}}, default=str))
"""
    raw_result = await shared_code_interpreter.run_code(code=code)
    return _stdout_from_ci_result(raw_result)


async def preview_template_file(path: str, max_rows: int = 10) -> str:
    """Preview a supported file from the E2B template."""
    normalized_path = str(PurePosixPath(path))
    suffix = PurePosixPath(normalized_path).suffix.lower()
    if suffix not in _SUPPORTED_PREVIEW_SUFFIXES:
        raise ValueError(
            f"Unsupported file type for preview: {suffix}. Supported: {sorted(_SUPPORTED_PREVIEW_SUFFIXES)}"
        )

    code = f"""
import json
import sqlite3
from itertools import islice
from pathlib import Path

import pandas as pd

path = Path({normalized_path!r})
max_rows = int({max_rows})
suffix = path.suffix.lower()

if not path.exists():
    raise FileNotFoundError(f"File not found: {{path}}")

if suffix in {{'.csv'}}:
    df = pd.read_csv(path, nrows=max_rows)
    payload = {{"path": str(path), "kind": "table", "preview": df.to_dict(orient='records')}}
elif suffix in {{'.parquet'}}:
    df = pd.read_parquet(path).head(max_rows)
    payload = {{"path": str(path), "kind": "table", "preview": df.to_dict(orient='records')}}
elif suffix in {{'.json'}}:
    with path.open() as handle:
        data = json.load(handle)
    if isinstance(data, list):
        preview = data[:max_rows]
    elif isinstance(data, dict):
        preview = dict(list(data.items())[:max_rows])
    else:
        preview = data
    payload = {{"path": str(path), "kind": "json", "preview": preview}}
elif suffix in {{'.jsonl'}}:
    with path.open() as handle:
        preview = [json.loads(line) for line in islice(handle, max_rows)]
    payload = {{"path": str(path), "kind": "jsonl", "preview": preview}}
elif suffix in {{'.db', '.sqlite', '.sqlite3'}}:
    conn = sqlite3.connect(path)
    rows = conn.execute(
        \"\"\"
        SELECT name, type
        FROM sqlite_master
        WHERE type IN ('table', 'view')
          AND name NOT LIKE 'sqlite_%'
        ORDER BY type, name
        \"\"\"
    ).fetchall()
    preview = []
    for name, relation_type in rows:
        columns = conn.execute(f'PRAGMA table_info("{{name}}")').fetchall()
        preview.append({{
            "name": name,
            "type": relation_type,
            "columns": [{{"name": col[1], "type": col[2]}} for col in columns],
        }})
    conn.close()
    payload = {{"path": str(path), "kind": "sqlite", "preview": preview}}
else:
    text = path.read_text(errors='ignore')
    payload = {{"path": str(path), "kind": "text", "preview": text[:4000]}}

print(json.dumps(payload, default=str))
"""
    raw_result = await shared_code_interpreter.run_code(code=code)
    return _stdout_from_ci_result(raw_result)


async def analyze_potential_fraud(limit: int = 20) -> str:
    """Rank potentially suspicious transactions from the raw template files."""
    safe_limit = max(1, min(int(limit), 100))
    code = """
import heapq
import json
from pathlib import Path

import pandas as pd

root = Path("__TEMPLATE_ROOT__")
transactions_path = root / 'transactions_data.csv'
cards_path = root / 'cards_data.csv'
users_path = root / 'users_data.csv'
mcc_path = root / 'mcc_codes.json'

cards = pd.read_csv(cards_path)
users = pd.read_csv(users_path)
with mcc_path.open() as handle:
    mcc_codes = json.load(handle)

cards = cards.rename(columns={'id': 'card_join_id'})
users = users.rename(columns={'id': 'user_join_id'})

transactions_usecols = [
    'id',
    'date',
    'client_id',
    'card_id',
    'amount',
    'use_chip',
    'merchant_id',
    'merchant_city',
    'merchant_state',
    'zip',
    'mcc',
    'errors',
]

amount_samples = []
chunk_size = 50000
max_rows_to_scan = 200000
rows_sampled = 0
for chunk in pd.read_csv(transactions_path, usecols=transactions_usecols, chunksize=chunk_size):
    rows_sampled += len(chunk)
    chunk['amount_numeric'] = (
        chunk['amount']
        .astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .astype(float)
    )
    amount_samples.extend(chunk['amount_numeric'].head(1000).tolist())
    if len(amount_samples) >= 50000 or rows_sampled >= max_rows_to_scan:
        break

if not amount_samples:
    raise RuntimeError('No transaction amounts were available for fraud analysis.')

sample_series = pd.Series(amount_samples, dtype='float64')
amount_threshold = float(sample_series.quantile(0.99))

high_risk_terms = [
    'money transfer',
    'wire transfer',
    'telecommunication',
    'travel',
    'miscellaneous',
    'digital',
    'online',
]

flagged_transactions = 0
total_transactions = 0
top_candidates = []
rows_processed = 0

def normalize_bool(series):
    return (
        series.fillna('')
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(['yes', 'true', '1', 'y'])
    )

for chunk in pd.read_csv(transactions_path, usecols=transactions_usecols, chunksize=chunk_size):
    total_transactions += len(chunk)
    rows_processed += len(chunk)
    chunk['amount_numeric'] = (
        chunk['amount']
        .astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .astype(float)
    )
    merged = chunk.merge(
        cards,
        left_on='card_id',
        right_on='card_join_id',
        how='left',
        suffixes=('', '_card'),
    )
    merged = merged.merge(
        users,
        left_on='client_id',
        right_on='user_join_id',
        how='left',
        suffixes=('', '_user'),
    )

    merged['mcc_description'] = merged['mcc'].astype(str).map(lambda value: mcc_codes.get(str(value), 'Unknown'))
    merged['errors_text'] = merged['errors'].fillna('').astype(str)
    merged['use_chip_text'] = merged['use_chip'].fillna('').astype(str)
    merged['merchant_state_text'] = merged['merchant_state'].fillna('').astype(str).str.upper()
    merged['card_on_dark_web_flag'] = normalize_bool(
        merged.get('card_on_dark_web', pd.Series(index=merged.index, dtype=object))
    )
    merged['high_amount_flag'] = merged['amount_numeric'] >= amount_threshold
    merged['error_flag'] = merged['errors_text'].str.strip().ne('')
    merged['online_flag'] = merged['use_chip_text'].str.contains('online', case=False, na=False)

    address_series = merged.get('address', pd.Series(index=merged.index, dtype=object)).fillna('').astype(str)
    state_match = address_series.str.extract(r',\\s*([A-Z]{2})\\s+\\d{5}(?:-\\d{4})?$', expand=False)
    merged['user_state_guess'] = state_match.fillna('').str.upper()
    merged['geo_mismatch_flag'] = (
        merged['user_state_guess'].ne('')
        & merged['merchant_state_text'].ne('')
        & merged['user_state_guess'].ne(merged['merchant_state_text'])
    )
    merged['high_risk_mcc_flag'] = merged['mcc_description'].str.lower().apply(
        lambda text: any(term in text for term in high_risk_terms)
    )

    merged['risk_score'] = (
        merged['card_on_dark_web_flag'].astype(int) * 5
        + merged['high_amount_flag'].astype(int) * 3
        + merged['geo_mismatch_flag'].astype(int) * 2
        + merged['error_flag'].astype(int) * 2
        + merged['online_flag'].astype(int) * 1
        + merged['high_risk_mcc_flag'].astype(int) * 1
    )

    flagged = merged[merged['risk_score'] > 0].copy()
    flagged_transactions += len(flagged)
    if flagged.empty:
        continue

    for _, row in flagged.iterrows():
        reasons = []
        if bool(row['card_on_dark_web_flag']):
            reasons.append('card_on_dark_web')
        if bool(row['high_amount_flag']):
            reasons.append('high_amount')
        if bool(row['geo_mismatch_flag']):
            reasons.append('state_mismatch')
        if bool(row['error_flag']):
            reasons.append('transaction_errors_present')
        if bool(row['online_flag']):
            reasons.append('online_transaction')
        if bool(row['high_risk_mcc_flag']):
            reasons.append('higher_risk_merchant_category')

        candidate = {
            'id': int(row['id']),
            'date': str(row['date']),
            'client_id': int(row['client_id']),
            'card_id': int(row['card_id']),
            'amount_numeric': float(row['amount_numeric']),
            'merchant_city': None if pd.isna(row['merchant_city']) else str(row['merchant_city']),
            'merchant_state': None if pd.isna(row['merchant_state']) else str(row['merchant_state']),
            'mcc': None if pd.isna(row['mcc']) else str(row['mcc']),
            'mcc_description': str(row['mcc_description']),
            'use_chip': None if pd.isna(row['use_chip']) else str(row['use_chip']),
            'errors': None if pd.isna(row['errors']) else str(row['errors']),
            'risk_score': int(row['risk_score']),
            'risk_reasons': reasons,
        }
        sort_key = (candidate['risk_score'], candidate['amount_numeric'], candidate['date'])
        if len(top_candidates) < __SAFE_LIMIT__:
            heapq.heappush(top_candidates, (sort_key, candidate))
        elif sort_key > top_candidates[0][0]:
            heapq.heapreplace(top_candidates, (sort_key, candidate))

    if rows_processed >= max_rows_to_scan:
        break

results = [item[1] for item in sorted(top_candidates, key=lambda item: item[0], reverse=True)]
payload = {
    'row_limit': __SAFE_LIMIT__,
    'total_transactions': int(total_transactions),
    'rows_scanned': int(rows_processed),
    'scan_limited': rows_processed >= max_rows_to_scan,
    'flagged_transactions': int(flagged_transactions),
    'high_amount_threshold': amount_threshold,
    'heuristics': {
        'card_on_dark_web': 'Card is marked as exposed or compromised.',
        'high_amount': 'Transaction amount is above the sampled 99th percentile threshold.',
        'state_mismatch': 'Merchant state differs from a state parsed from the user address.',
        'transaction_errors_present': 'Transaction has a non-empty errors field.',
        'online_transaction': 'Transaction channel is online.',
        'higher_risk_merchant_category': 'Merchant category description contains a higher-risk keyword.',
    },
    'results': results,
}
print(json.dumps(payload, default=str))
"""
    code = code.replace("__TEMPLATE_ROOT__", _TEMPLATE_ROOT)
    code = code.replace("__SAFE_LIMIT__", str(safe_limit))
    raw_result = await shared_code_interpreter.run_code(code=code)
    return _stdout_from_ci_result(raw_result)


TEMPLATE_AWARE_AGENT_INSTRUCTIONS = f"""
You are a template-aware analytics agent running against files available in the E2B sandbox.

Available tools:
- list_template_files: list all available files under {_TEMPLATE_ROOT}
- preview_template_file: inspect non-database files or inspect schemas for SQLite files
- analyze_potential_fraud: rank suspicious transactions from the raw fraud-analysis files

Behavior rules:
- Start with list_template_files when you are unsure which files exist.
- Use preview_template_file on newly available files before deciding how to answer.
- If the user asks for suspicious or potentially fraudulent transactions from the raw files, call analyze_potential_fraud.
- Prefer file-based reasoning and pandas-style analysis for raw CSV/JSON data.
- Never invent file names, columns, or tables.
- Return the answer directly for natural-language questions.
- Keep the answer concise and grounded in tool output.
- Include a short section naming which files you used.
""".strip()


template_aware_agent = Agent(
    name="template_aware_e2b_agent",
    model=AGENT_LLM_NAMES["worker"],
    instruction=TEMPLATE_AWARE_AGENT_INSTRUCTIONS,
    tools=[
        list_template_files,
        preview_template_file,
        analyze_potential_fraud,
    ],
)

template_aware_session_service = InMemorySessionService()
template_aware_runner = Runner(
    app_name=template_aware_agent.name,
    agent=template_aware_agent,
    session_service=template_aware_session_service,
)
template_aware_session = await template_aware_session_service.create_session(
    app_name=template_aware_agent.name,
    user_id="notebook-user",
    state={},
)


async def reset_template_aware_agent_session() -> str:
    """Reset the session for the template-aware agent."""
    global template_aware_session
    template_aware_session = await template_aware_session_service.create_session(
        app_name=template_aware_agent.name,
        user_id="notebook-user",
        state={},
    )
    return template_aware_session.id


async def answer_template_query(question: str) -> str:
    """Answer a natural-language question using the current E2B template files."""
    content = types.Content(role="user", parts=[types.Part(text=question)])
    final_chunks: list[str] = []

    async for event in template_aware_runner.run_async(
        user_id="notebook-user",
        session_id=template_aware_session.id,
        new_message=content,
    ):
        if not event.is_final_response() or not event.content:
            continue
        for part in event.content.parts:
            if part.text:
                final_chunks.append(part.text)

    return "\n".join(chunk for chunk in final_chunks if chunk).strip()


template_file_catalog = await list_template_files()
print(template_file_catalog)
print(f"Template-aware agent ready. Session: {template_aware_session.id}")

{"root": "/data", "files": [{"path": "/data/cards_data.csv", "name": "cards_data.csv", "suffix": ".csv", "size_bytes": 509860}, {"path": "/data/mcc_codes.json", "name": "mcc_codes.json", "suffix": ".json", "size_bytes": 4716}, {"path": "/data/train_fraud_labels.json", "name": "train_fraud_labels.json", "suffix": ".json", "size_bytes": 159083088}, {"path": "/data/transactions_data.csv", "name": "transactions_data.csv", "suffix": ".csv", "size_bytes": 1258531040}, {"path": "/data/users_data.csv", "name": "users_data.csv", "suffix": ".csv", "size_bytes": 164831}]}
Template-aware agent ready. Session: a5d2011f-59ed-4c28-93b8-08f0f12f7df7


In [22]:
example_template_response = await answer_template_query(
    "Using the new template files, summarize what data is available and explain how transactions_data.csv, cards_data.csv, users_data.csv, and mcc_codes.json could be combined for fraud analysis."
)
print(example_template_response)

21:06:06.920 invocation
21:06:06.922   invoke_agent template_aware_e2b_agent
21:06:06.924     call_llm
21:06:08.071       execute_tool list_template_files


21:06:09.487     call_llm
21:06:10.760       execute_tool preview_template_file


21:06:11.746     call_llm
21:06:12.777       execute_tool preview_template_file


21:06:14.621     call_llm
21:06:16.071       execute_tool preview_template_file


21:06:18.083     call_llm
21:06:19.156       execute_tool preview_template_file


21:06:20.824     call_llm
The available data includes `transactions_data.csv`, `cards_data.csv`, `users_data.csv`, and `mcc_codes.json`.

**Data Summary:**

*   **`transactions_data.csv`**: Contains individual transaction records, including `id`, `date`, `client_id`, `card_id`, `amount`, `use_chip`, `merchant_id`, `merchant_city`, `merchant_state`, `zip`, `mcc`, and `errors`.
*   **`cards_data.csv`**: Details card-specific information like `id` (card ID), `client_id`, `card_brand`, `card_type`, `card_number`, `expires`, `cvv`, `has_chip`, `num_cards_issued`, `credit_limit`, `acct_open_date`, `year_pin_last_changed`, and `card_on_dark_web`.
*   **`users_data.csv`**: Provides client (user) demographics and financial data, such as `id` (client ID), `current_age`, `retirement_age`, `birth_year`, `birth_month`, `gender`, `address`, `latitude`, `longitude`, `per_capita_income`, `yearly_income`, `total_debt`, `credit_score`, and `num_credit_cards`.
*   **`mcc_codes.json`**: A dictionary mappi

In [23]:
await reset_template_aware_agent_session()
example_template_response = await answer_template_query(
    "What files are available in /data, and which files would you use to join transactions to cardholders for fraud analysis?"
)
print(example_template_response)

21:06:27.075 invocation
21:06:27.076   invoke_agent template_aware_e2b_agent
21:06:27.077     call_llm
21:06:28.405       execute_tool list_template_files


21:06:29.984     call_llm
21:06:31.333       execute_tool preview_template_file


21:06:33.500     call_llm
21:06:34.519       execute_tool preview_template_file


21:06:36.287     call_llm
21:06:37.644       execute_tool preview_template_file


21:06:38.660     call_llm
The files available in `/data` are:
* `cards_data.csv`
* `mcc_codes.json`
* `train_fraud_labels.json`
* `transactions_data.csv`
* `users_data.csv`

To join transactions to cardholders for fraud analysis, I would use the following files:
*   `transactions_data.csv` (contains `client_id` and `card_id`)
*   `cards_data.csv` (contains `id` which can be joined with `transactions_data.csv` on `card_id`, and `client_id`)
*   `users_data.csv` (contains `id` which can be joined with `cards_data.csv` on `client_id`)


In [24]:
await reset_template_aware_agent_session()
example_template_response = await answer_template_query(
    "Find potentially fraudulent transactions using the available files. Prioritize cards marked on the dark web, unusually large purchases, and transactions that look geographically inconsistent with the user profile."
)
print(example_template_response)

21:06:41.518 invocation
21:06:41.519   invoke_agent template_aware_e2b_agent
21:06:41.520     call_llm
21:06:42.695       execute_tool analyze_potential_fraud


21:06:49.094     call_llm
Here are the top 20 potentially fraudulent transactions, ranked by risk score. The analysis prioritized cards on the dark web, unusually large purchases, and geographically inconsistent transactions.

**Heuristics Used for Fraud Detection:**
*   **card_on_dark_web**: Card is marked as exposed or compromised.
*   **high_amount**: Transaction amount is above the sampled 99th percentile threshold ($300.48).
*   **state_mismatch**: Merchant state differs from a state parsed from the user address.
*   **transaction_errors_present**: Transaction has a non-empty errors field.
*   **online_transaction**: Transaction channel is online.
*   **higher_risk_merchant_category**: Merchant category description contains a higher-risk keyword.

**Top Potentially Fraudulent Transactions:**

1.  **ID: 7612465**
    *   **Date:** 2010-02-05 01:20:00
    *   **Amount:** $356.88
    *   **Merchant:** ONLINE (Travel Agencies)
    *   **Errors:** Bad Card Number
    *   **Risk Score:*

In [25]:
await reset_template_aware_agent_session()
example_template_response = await answer_template_query(
    "Show the top 20 most suspicious transactions and include the risk factors used."
)
print(example_template_response)

21:06:58.719 invocation
21:06:58.720   invoke_agent template_aware_e2b_agent
21:06:58.722     call_llm
21:06:59.875       execute_tool analyze_potential_fraud


21:07:08.423     call_llm
Here are the top 20 most suspicious transactions:

| Transaction ID | Risk Score | Risk Reasons                                                                   |
|----------------|------------|--------------------------------------------------------------------------------|
| 7612465        | 7          | High amount, Transaction errors present, Online transaction, Higher risk merchant category |
| 7510376        | 7          | High amount, Transaction errors present, Online transaction, Higher risk merchant category |
| 7507721        | 7          | High amount, Transaction errors present, Online transaction, Higher risk merchant category |
| 7536966        | 6          | High amount, Transaction errors present, Online transaction                    |
| 7522752        | 6          | High amount, Transaction errors present, Online transaction                    |
| 7668064        | 6          | High amount, Transaction errors present, Higher risk merchant ca